In [ ]:
import os
os.environ["XLA_FLAGS"] = '--xla_force_host_platform_device_count=8'

In [ ]:
from jax import numpy as jnp
from jax import lax, jit, make_jaxpr
import jax

import pandas as pd

import numpyro
numpyro.set_host_device_count(4)

In [ ]:
jax.device_count()

In [ ]:
jax.config.update("jax_enable_x64", True)

In [ ]:
DTYPE = jnp.float32

f64 = jnp.float64
f32 = jnp.float32

In [ ]:
def get_densities(window_len, dtype=DTYPE):
    x = jnp.linspace(0.0, 1.0, window_len, dtype=dtype)
    triangle = x * x[::-1]
    return triangle / triangle.sum()

In [ ]:
from jax.scipy.stats import gamma as jaxgamma

def get_gamma_densities(window_len, mean, sd):
    var = sd ** 2.0
    scale = var / mean
    a = mean / scale
    cum_dens = jaxgamma.cdf(jnp.arange(window_len + 1), a=a,scale=scale)
    diff_series = jnp.diff(cum_dens)
    return diff_series/diff_series.sum()


In [ ]:
pd.Series(get_gamma_densities(32, 8.0,5.5)).plot()

In [ ]:
from jaxtyping import Array, Float, PyTree

In [ ]:
from typing import NamedTuple

class RenewalState(NamedTuple):
    suscept: float
    incidence: jax.typing.ArrayLike

class RenewalStateP(NamedTuple):
    suscept: int
    incidence: jax.typing.ArrayLike

class RenewResults(NamedTuple):
    outputs: dict[str, jax.typing.ArrayLike]
    full_incidence: Float[Array, "full_times"]

In [ ]:
from numpyro import distributions as dist
from jax.random import PRNGKey, split

In [ ]:
def renew(contact_rate, ntimes, window_len, dtype, stochastic=False):
    densities = get_densities(window_len, dtype)
    init_incidence = get_gamma_densities(window_len, gmean, gsd)
    init_state = RenewalState(1000.0, init_incidence)

    def state_update(state: RenewalState, t) -> tuple[RenewalState, jnp.array]:
        target_inc = (densities * state.incidence).sum() * contact_rate  # Calculated incidence
        actual_inc = jnp.minimum(target_inc, state.suscept)  # Incidence after ceiling applied
        suscept = state.suscept - actual_inc  # Susceptible depletion
        inc = jnp.concat([jnp.array([actual_inc]), state.incidence[:-1]])  # Move up in matrix
        out = {"incidence": actual_inc, "suscept": suscept}
        return RenewalState(suscept, inc), out

    end_state, outputs = lax.scan(state_update, init_state, jnp.arange(ntimes))
    full_inc = jnp.concatenate([init_incidence, jnp.array(outputs["incidence"])])
    return RenewResults(outputs, full_incidence=full_inc)


In [ ]:
def get_inc(densities, state, contact_rate, ipop, k):
    target_inc = (densities * state.incidence).sum() * contact_rate * (state.suscept/ipop)  # Calculated incidence
    target_inc = dist.Poisson(target_inc).sample(k)
    actual_inc = jnp.minimum(target_inc, state.suscept) # Incidence after ceiling applied
    return actual_inc.astype(jnp.int32)

def ret0(*args):
    return jnp.int32(0)

def renewp(k, cr0, cr1, gmean, gsd, ipop, ntimes, window_len, dtype):
    densities = get_gamma_densities(window_len, gmean, gsd)
    init_incidence = jnp.ones(window_len, dtype=jnp.int32)
    init_state = RenewalStateP(ipop, init_incidence)

    keys = split(k, ntimes)

    def state_update(state: RenewalStateP, t) -> tuple[RenewalState, jnp.array]:

        contact_rate = t/ntimes * cr0 + (ntimes-t)/ntimes * cr1

        actual_inc = lax.cond(state.suscept * state.incidence.sum() == 0, ret0, get_inc, densities, state, contact_rate, ipop, keys[t])

        #target_inc = (densities * state.incidence).sum() * contact_rate  # Calculated incidence
        #target_inc = dist.Poisson(target_inc, is_sparse=True).sample(keys[t])
        #actual_inc = jnp.minimum(target_inc, state.suscept) # Incidence after ceiling applied
        suscept = state.suscept - actual_inc  # Susceptible depletion
        inc = jnp.concat([jnp.array([actual_inc]), state.incidence[:-1]])  # Move up in matrix
        out = {"incidence": actual_inc, "suscept": suscept}
        return RenewalStateP(suscept, inc), out

    end_state, outputs = lax.scan(state_update, init_state, jnp.arange(ntimes))
    full_inc = jnp.concatenate([init_incidence, jnp.array(outputs["incidence"])])
    return RenewResults(outputs, full_incidence=full_inc)


In [ ]:
2/1000

In [ ]:
wlen = 32
cr = 1.
gmean = 8.0
gsd = 2.0
ntimes = 200
ipop = 10000
k = PRNGKey(0)

rj = jit(renewp, static_argnames=["ntimes", "window_len", "dtype"])

#s0 = rj(k, cr, ntimes, wlen, f32).outputs["suscept"]
#s1 = rj(PRNGKey(1), cr, ntimes, wlen, f32).outputs["suscept"]
#s2 = rj(PRNGKey(2), cr, ntimes, wlen, f32).outputs["suscept"]

In [ ]:
import numpyro

In [ ]:
import numpy as np

In [ ]:
mu = 2.0

In [ ]:
thing = pd.Series(dist.Normal(mu,np.sqrt(mu)).sample(k,(1000000,)))

In [ ]:
def qgamma(rate, k):

    tfunc = lambda rate,k: dist.Gamma(jnp.where(rate>0.0,rate,1e-32)).sample(k)
    ffunc = lambda rate, k: dist.TruncatedNormal(rate,jnp.sqrt(rate),low=0.0).sample(k)
    return lax.cond(rate<10.0, tfunc, ffunc, rate,k)

In [ ]:
jqg = jit(qgamma)

In [ ]:
 init_incidence = get_gamma_densities(window_len, gmean, gsd)
    init_state = RenewalState(1000.0, init_incidence)

    def state_update(state: RenewalState, t) -> tuple[RenewalState, jnp.array]:
        target_inc = (densities * state.incidence).sum() * contact_rate  # Calculated incidence
        actual_inc = jnp.minimum(target_inc, state.suscept)  # Incidence after ceiling applied
        suscept = state.suscept - actual_inc  # Susceptible depletion
        inc = jnp.concat([jnp.array([actual_inc]), state.incidence[:-1]])  # Move up in matrix
        out = {"incidence": actual_inc, "suscept": suscept}
        return RenewalState(suscept, inc), out

In [ ]:
def _qsample(k, contact_rate, gmean, gsd, n):

    densities = get_gamma_densities(50, gmean, gsd)

    def scanf(state, x):
        suscept, incidence, state_k = state
        curk, curk2, nextk = split(state_k,3)
        iprop = suscept / (10000.0)

        dinc = (densities * incidence).sum() * contact_rate

        new_inf = qgamma(iprop*contact_rate*dinc, curk)

        new_inf = jnp.minimum(new_inf, suscept)

        suscept = suscept-new_inf

        inc = jnp.concat([jnp.array([new_inf]), incidence[:-1]])

        return (suscept, incidence, nextk), (new_inf)

    _, out = lax.scan(scanf, (9990.0,densities*10.0,k), jnp.arange(n))
    return out

qsample = jit(_qsample, static_argnames=["n"])

In [ ]:
def _qsamplesir(k, contact_rate, recovery_rate, n):

    def scanf(state, x):
        suscept, infected, state_k = state
        curk, curk2, nextk = split(state_k,3)
        iprop = suscept / (10000.0)
        new_inf = qgamma(iprop*contact_rate*infected, curk)

        new_inf = jnp.minimum(new_inf, suscept)

        recovered = qgamma(infected * recovery_rate, curk2)

        recovered = jnp.minimum(infected, recovered)

        suscept = suscept-new_inf
        infected = infected+new_inf-recovered
        infected = jnp.where(infected<1.0, 0.0, infected)

        return (suscept, infected, nextk), (new_inf)

    _, out = lax.scan(scanf, (9990.0,10.0,k), jnp.arange(n))
    return out

qsample = jit(_qsample, static_argnames=["n"])

In [ ]:
def thing(k, contact_rate, recovery_rate, n):
    return jnp.cumsum(qsample(k, contact_rate, recovery_rate, n))[-1]

In [ ]:
tgrad = jax.grad(thing, [1,2])

In [ ]:
target_params = [0.3,0.2]

In [ ]:
vqsample = jax.vmap(qsample, [0,None,None,None])
vjq = jax.jit(vqsample, static_argnames=["n"])

In [ ]:
k = PRNGKey(6)
x = vjq(split(k,20),0.3,0.2,200)

In [ ]:
target_params = [0.41,0.3]
k = PRNGKey(16)
obs = qsample(k, *target_params, 200)
samples = vqsample(split(k,4),*target_params,200)
pd.DataFrame(samples.T).plot(legend=False)
pd.Series(obs).plot(color="black")

In [ ]:
dist.Gamma(samples+1e-32).log_prob(obs+1e-32).mean()

In [ ]:
from numpyro import infer

In [ ]:
vqsample

In [ ]:
def npmodel(k):
    contact_rate = numpyro.sample("contact_rate", dist.Uniform(0.001, 2.0))
    recovery_rate = numpyro.sample("recovery_rate", dist.Uniform(0.001, 2.0))
    res = vqsample(split(k,4), contact_rate, recovery_rate, 200)

    ll = numpyro.factor("ll", dist.Gamma(res+1e-32).log_prob(obs+1e-32).mean())
    

In [ ]:
minfo = infer.util.initialize_model(k, npmodel, dynamic_args=True, model_args=(k,))

In [ ]:

def pfstep(carry, i):
    ki = carry
    nextki, curki = split(ki,2)
    pe = minfo.potential_fn(curki)({"contact_rate": 0.4, "recovery_rate": 0.3})
    return nextki, pe

nextk, pe = lax.scan(pfstep, k, jnp.arange(10))

In [ ]:
%%time
nextk, pe = lax.scan(pfstep, k, jnp.arange(20))

In [ ]:
import optax

In [ ]:
guide = infer.autoguide.AutoNormal(npmodel)
svi = infer.SVI(npmodel, guide, optax.adam(1e-1), infer.Trace_ELBO())

In [ ]:
sstate = svi.init(k,k)

In [ ]:
svires = svi.run(k,2000,k)

In [ ]:
pd.Series(svires.losses).plot()

In [ ]:
svires.params

In [ ]:
psamples = guide.sample_posterior(k, svi.get_params(final_state), sample_shape=(256,))

In [ ]:
psamples = guide.sample_posterior(k, svi.get_params(final_state), sample_shape=(256,))
pd.DataFrame(psamples).hist()

In [ ]:
guide = infer.autoguide.AutoNormal(npmodel)
svi = infer.SVI(npmodel, guide, optax.adam(1e-1), infer.Trace_ELBO())

@jit
def runsvi(k):
    sstate = svi.init(k,k)

    ks = split(k,1000)

    def inner(carry, i):
        sstate = carry
        sstate, loss = svi.update(sstate, ks[i])
        return sstate, loss
    
    final_state, losses = lax.scan(inner, sstate, jnp.arange(1000))

    return final_state, losses

In [ ]:
final_state, losses = runsvi(PRNGKey(1))

In [ ]:
psamples = guide.sample_posterior(k, svi.get_params(final_state), sample_shape=(256,))
pd.DataFrame(psamples).hist()

In [ ]:
target_params = [0.4,0.3]
k = PRNGKey(6)
obs = qsample(k, *target_params, 200)
samples = vqsample(split(k,10),*target_params,200)
pd.DataFrame(samples.T).plot(legend=False)
pd.Series(obs).plot(color="black")

In [ ]:
def get_samples(k, params, ens_size):
    samples = vqsample(split(k,ens_size),*params,200)
    return samples

In [ ]:
get_samples(PRNGKey(10), [0.15,0.1], 4).mean(axis=1).mean()

In [ ]:
get_samples(PRNGKey(10), [0.6,0.4], 4).std(axis=1).mean()

In [ ]:
get_samples(PRNGKey(10), [0.6,0.4], 4).max(axis=1).mean()

In [ ]:
samples.mean(axis=1).mean()

In [ ]:
samples.std(axis=1).mean()

In [ ]:
samples.max(axis=1).mean()

In [ ]:
eucdist_ensemble(obs, samples).mean()

In [ ]:
def eucdist(x,y):
    return ((x-y)*(x-y)).sum()

In [ ]:
def compute_loss(k, params, n):
    contact_rate, recovery_rate = params[0], params[1]
    return eucdist(obs, qsample(k, contact_rate, recovery_rate, n))

In [ ]:
def eucdist_ensemble(obs, ens):
    diff = obs-ens
    return (diff*diff).sum(axis=1)

In [ ]:
from numpyro import distributions as dist

In [ ]:
ks = split(k, 10)
ens_res = vqsample(ks, *target_params, 200)

In [ ]:
pd.Series(ens_res[0,:]).plot()
pd.Series(obs).plot(color="black")

In [ ]:
ens_res.mean(axis=1)

In [ ]:
pd.DataFrame(ens_res.cumsum(axis=1).T).plot()

In [ ]:
def compute_loss_ensemble(k, params, n, ensemble_size):
    ks = split(k, ensemble_size)
    contact_rate, recovery_rate = params[0], params[1]

    #def scanf(carry, i):
    #    return carry, eucdist(obs, qsample(ks[i], contact_rate, recovery_rate, n))

    #dists = [eucdist(obs, qsample(ks[i], contact_rate, recovery_rate, n)) for i in range(10)]
    #_, dists = lax.scan(scanf, None, xs=jnp.arange(10))
    ens = vqsample(ks, contact_rate, recovery_rate, n)
    #dists = eucdist_ensemble(obs, ens)
    #dists = 0.0 - (dist.Gamma(obs+1e-32).log_prob(ens+1e-32)).sum(axis=1)
    #dists = eucdist_ensemble(obs.cumsum(), ens.cumsum(axis=1))
    dist_mean = (obs.mean() - ens.mean(axis=1).mean())**2
    #dist_mean = (diff_mean * diff_mean).
    dist_std = (obs.std() - ens.std(axis=1).mean())**2
    return dist_mean + dist_std

In [ ]:
compute_loss_ensemble = jit(compute_loss_ensemble, static_argnames=["n", "ensemble_size"])

In [ ]:
gloss = jax.grad(compute_loss_ensemble, [1])

In [ ]:
import optax

In [ ]:
from math import log
from numpy import exp


def logit(u):
    return jnp.log(u / (1 - u))


def inverse_logit(v):
    return 1.0 / (1.0 + jnp.exp(-v))

In [ ]:

lower_bound = 0.0001
upper_bound = 1.0

def unconstrain(x):
  return logit((x - lower_bound) / (upper_bound - lower_bound))

def constrain(x):
  return lower_bound + (upper_bound - lower_bound) * inverse_logit(x)

In [ ]:
cur_loss = np.inf
best_params = None

for i in range(4):
    cparams = np.random.uniform(lower_bound, upper_bound, size=(2,))
    loss = compute_loss_ensemble(k, cparams, 200, 16)
    if loss < cur_loss:
        cur_loss = loss
        best_params = cparams

best_params

In [ ]:
ensemble_size = 4

start_learning_rate = 1e-1
optimizer = optax.adam(start_learning_rate, nesterov=True)
#optimizer = optax.noisy_sgd(start_learning_rate, eta=1e-5)

# Initialize parameters of the model + optimizer.
cparams = best_params#jnp.array([0.8,0.8])

params = unconstrain(cparams)


opt_state = optimizer.init(params)

cur_loss = np.inf
best_params = cparams

k = PRNGKey(5)
losses = []
sampled_params = []
for __ in range (40):
  for _ in range(40):
    k, lossk = split(k, 2)
    grads = gloss(lossk, cparams, 200, ensemble_size)[0]
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    #params = optax.projections.projection_box(params, 0.0001, 2.0)
    cparams = constrain(params)

    loss = compute_loss_ensemble(lossk, cparams, 200, ensemble_size)
    losses.append(loss)
    sampled_params.append(cparams)
    if loss < cur_loss:
      cur_loss = loss
      best_params = cparams
  #print(compute_loss_ensemble(k, params, 200))
  print(cur_loss, loss, best_params, cparams, target_params)

losses = pd.Series(np.array(losses))

losses.plot()

In [ ]:
spdf = pd.DataFrame(np.array(sampled_params))

In [ ]:
losses[losses < losses.mean() * 0.5].plot()

In [ ]:
spdf[losses < losses.mean() * 0.5].plot()

In [ ]:
pdf = pd.DataFrame(np.array(sampled_params))

meanp = pdf.mean()
pdf.hist()#plot(kind="hist", weights=inv_losses)

In [ ]:
compute_loss_ensemble(k, best_params, 200, 100), compute_loss_ensemble(k, target_params, 200, 100)

In [ ]:
pd.Series(obs).plot()

In [ ]:
pd.DataFrame(vqsample(split(k,400),*best_params,200).T).plot(legend=False)
pd.Series(obs).plot(color="black")

In [ ]:
params

In [ ]:
grads = []
for k in split(PRNGKey(0),50):
    cr, rr = tgrad(k, 0.1, 0.1, 200)
    grads.append(np.array([cr,rr]))

pd.DataFrame(np.array(grads)).hist()

In [ ]:
pd.Series(qsample(k, 0.1, 0.08, 800)).cumsum().plot()

In [ ]:
optax.projections.projection_non_negative(params)

In [ ]:
for k in split(PRNGKey(0),50):
    pd.Series(qsample(k, 0.1, 0.08, 800)).cumsum().plot()

In [ ]:
%%timeit
thing = qsample(k,10.0, 20.0, 1000)

In [ ]:
pd.Series(dist.Gamma(mu).sample(k, (1000000,))).hist()

In [ ]:
thing.mask(thing<0.0, 0.0).hist()

In [ ]:
pd.Series(dist.Poisson(mu).sample(k,(1000000,))).hist()

In [ ]:
cases0 = dist.Binomial(i0.astype(jnp.int64), probs=0.3).sample(PRNGKey(2))

pd.Series(i0).plot()
pd.Series(sres).plot()

In [ ]:
i0 = rj(PRNGKey(12), 5.7, 0.8, 4.5, 4.4, ipop, ntimes, wlen, f32).outputs["incidence"]

pd.Series(i0).plot()

In [ ]:
def thing(x,k):
    return dist.Poisson(x).sample(k)

In [ ]:
from jax import grad, value_and_grad,vjp

In [ ]:
primals, f_vjp = jax.vjp(thing, 10.0, k)

In [ ]:
thing(10.0,k)

In [ ]:
primals

In [ ]:
f_vjp(25.0)

In [ ]:
grad(thing, allow_int=True)(5.0,k)

In [ ]:
dist.Poisson(1e-10).log_prob(0)

In [ ]:
def npmodel():
    c0 = numpyro.sample("c0", dist.Uniform(0.5,20.0))
    c1 = numpyro.sample("c1", dist.Uniform(0.5,20.0))
    gmean = numpyro.sample("gmean", dist.TruncatedNormal(6.5,1.0, low=2.0, high=12.0))#, low=1.0, high=20.0))
    gsd = numpyro.sample("gsd", dist.Uniform(2.0,6.0))

    k = numpyro.prng_key()

    s = renewp(k, c0, c1, gmean, gsd, ipop, ntimes, wlen, f32).outputs["incidence"]

    #cases = numpyro.sample("cases", dist.Binomial(s, probs=0.3))
    numpyro.deterministic("k", k)

    numpyro.factor("lp", dist.Poisson(s+1e-10).log_prob(i0).mean())

    

In [ ]:
with numpyro.handlers.seed(rng_seed=1):
    a = npmodel()
a

In [ ]:
from numpyro.infer.util import initialize_model
model_info = initialize_model(k, npmodel, dynamic_args=True)

In [ ]:
ucparams = model_info.param_info.z
#{"c0": 1.0, "c1": 2.0, "gmean": 5.0, "gsd": 2.0}


with numpyro.handlers.seed(rng_seed=1):
    cparams = numpyro.infer.util.constrain_fn(npmodel, (),{},ucparams)
cparams

In [ ]:
def param_map(*args):
    pkeys = ["c0","c1","gmean","gsd"]
    return {k:v for k,v in zip(pkeys,args)}

In [ ]:
import numpy as np

In [ ]:
def perturb(params, sd=0.1):
    return {k:v*np.exp(np.random.normal(0.0,sd)) for k,v in params.items()}

In [ ]:
perturb(ucparams)

In [ ]:
cparams = param_map(5.7,0.8,4.5,4.4)

with numpyro.handlers.seed(rng_seed=1):
    ucparams = numpyro.infer.util.unconstrain_fn(npmodel, (),{},cparams)
ucparams

In [ ]:
minfo = initialize_model(PRNGKey(2), npmodel, dynamic_args=True)

In [ ]:
print(-minfo.potential_fn()(pucp))

with numpyro.handlers.seed(rng_seed=1):
    cparams = numpyro.infer.util.constrain_fn(npmodel, (),{},pucp)
cparams

In [ ]:
k = PRNGKey(2)
s = renewp(k, cparams["c0"], cparams["c1"], cparams["gmean"], cparams["gsd"], ipop, ntimes, wlen, f32).outputs["incidence"]

#cases = numpyro.sample("cases", dist.Binomial(s, probs=0.3))

dist.Poisson(s+1e-10).log_prob(i0).mean()

In [ ]:
model_info.potential_fn()({"c0": 1.0, "c1": 2.0, "gmean": 5.0, "gsd": 2.0})

In [ ]:
kernel = numpyro.infer.SA(npmodel)#, max_steps=1000, max_iter=1000)

In [ ]:
mcmc = numpyro.infer.MCMC(kernel, num_warmup=10, num_samples=10, num_chains=1)#, progress_bar=True)

mcmc.run(PRNGKey(4))

In [ ]:
idata = mcmc.get_samples(group_by_chain=True)

In [ ]:
idata

In [ ]:
import arviz as az

In [ ]:
az.plot_trace(idata, compact=False);

In [ ]:
az.summary(idata)

In [ ]:
pd.Series(mcmc.get_samples()["gmean"]).hist()

In [ ]:
wlen = 32
cr = 1.2
ntimes = 10000

s32 = renew(cr, ntimes, wlen, f32).outputs["suscept"]
s64 = renew(cr, ntimes, wlen, f64).outputs["suscept"]

In [ ]:
renewj = jit(renew, static_argnames=["ntimes", "window_len", "dtype"])

In [ ]:
wlen = 32
cr = 1.2
ntimes = 10000

s32 = renewj(cr, ntimes, wlen, f32).outputs["suscept"]
s64 = renewj(cr, ntimes, wlen, f64).outputs["suscept"]

In [ ]:
%%timeit
s32 = renewj(cr, ntimes, wlen, f32).outputs["suscept"]

In [ ]:
%%timeit
s64 = renewj(cr, ntimes, wlen, f64).outputs["suscept"]

In [ ]:
pd.Series(s64).plot()